# Module 04: Dictionaries, Deeper

In Module 02 I learned how to create dictionaries, access keys with square brackets, and add new entries. This module builds on that — going into how Python actually *uses* dictionaries when you write real code, not toy examples.

On 2026-05-27, Claude walked me through a function called `extract_text` that's part of an ingester I'm building to load my Claude Code session JSONLs into my memory database. That function is FULL of dictionary patterns I hadn't met yet — `.get()` with defaults, nested access, defensive lookups. This module unpacks those patterns one at a time.

## New for this module: a different learning loop

Each lesson follows the same structure:
1. Read a small piece of code
2. **Predict** what it will return *before running it* — write the prediction in a markdown cell, no peeking
3. Run it and check what actually happened
4. Modify something small to see what changes
5. Write a "what I learned" cell in my own words
6. If something doesn't make sense, ask Claude inline using `print_llm_response("...")`

The point isn't to be right on the predictions. It's to be *honest about what I expected versus what happened* — that's where real understanding gets built.

*Setup: import helpers*

In [1]:
from llm_helpers import print_llm_response, get_llm_response

# Lesson 1: `dict[key]` vs `dict.get(key)`

In Module 2 I accessed dictionary values with square brackets, like `food["spice_level"]`. That works — but it has a hidden trap. This lesson is about what that trap is, and why `.get()` is the safer way to look up dictionary values in real code.

In [2]:
learning_today = {
    "module": 4,
    "topic": "dictionaries",
    "lessons_completed": 0,
    "feelings": "tired but engaged"
}

The two lines below are pulling values from the above dictionary. The first one will pull the value from the key "module" and the second one will throw an error message because that key does not exist in the dictionary.

In [3]:
learning_today["module"]
learning_today["favorite_color"]

KeyError: 'favorite_color'

In [4]:
print(learning_today["module"])
print(learning_today["favorite_color"])

4


KeyError: 'favorite_color'

The print command was able to print the value 4 because it's in the dictionary, but again, was unsuccessful at pulling the dict command "favorite_color" because of the KeyError - it's not a key currently in the dictionary. One thing to note here is that this error will stop all over code after it from running.

The lines below modify the dict command with a .get( ), so that when they pull, if the dictionary doesn't have that key, it will pull "I don't have one set yet" as the value for that command.

In [5]:
learning_today.get("module")
learning_today.get("favorite_color")
learning_today.get("favorite_color", "I don't have one set yet")

"I don't have one set yet"

In [6]:
print(learning_today.get("module"))
print(learning_today.get("favorite_color"))
print(learning_today.get("favorite_color", "I don't have one set yet"))

4
None
I don't have one set yet


*Reflection*

- What `.get()` returns when the key exists
- What it returns when the key's missing (no default)
- What it returns when the key's missing (with default)
- The RETURN vs DISPLAY distinction (briefly — one sentence)


**Synthesis cell — Lesson 1 takeaway.** A short rule-of-thumb in your own words: *when do I use `[]` vs `.get()`, and why?* Bonus: one line connecting back to `extract_text` (the function uses `.get()` everywhere — why?).

# Lesson 2: Iterating Over Dictionaries

Dictionaries aren't just for single lookups. Often I need to walk through *all* the key-value pairs — for example, when I want to print every entry, transform every value, or filter based on something. Python gives me three different ways to iterate, and choosing the right one matters.

First, the working dictionary for this lesson:

In [ ]:
me = {
    "name": "Jen",
    "current_module": 4,
    "favorite_color": "purple",
    "learning_python_since": "January 2026",
    "essays_published": 1
}

### 2a — Default iteration: just the keys

When you write `for x in some_dict:`, Python gives you just the **keys**, one at a time.

**🔮 Predict** — what do I think the next code cell will do?

*(write your prediction here, in plain English, before running the code)*

In [ ]:
for key in me:
    print(key)

**📝 Reflect** — what did I actually see, and what did I learn?

*(write your reflection here, in your own words, after running)*

### 2b — `.keys()` does the same thing, but explicitly

`.keys()` is the explicit way to ask for keys. It's identical in behavior to the default loop above, but more readable — anyone reading your code immediately knows you wanted keys.

**🔮 Predict** — what do I think the next code cell will do?

*(write your prediction here, in plain English, before running the code)*

In [ ]:
for key in me.keys():
    print(key)

**📝 Reflect** — what did I actually see, and what did I learn?

*(write your reflection here, in your own words, after running)*

### 2c — `.values()` gives you the values instead

What if I want the *values* without their keys?

**🔮 Predict** — what do I think the next code cell will do?

*(write your prediction here, in plain English, before running the code)*

In [ ]:
for value in me.values():
    print(value)

**📝 Reflect** — what did I actually see, and what did I learn?

*(write your reflection here, in your own words, after running)*

### 2d — `.items()` gives you BOTH keys and values

Most of the time, I want both. `.items()` gives me each key-value pair as a tuple — two variables at once. Notice the loop variable changes shape: instead of one name, I write two (`key, value`) separated by a comma.

**🔮 Predict** — what do I think the next code cell will do?

*(write your prediction here, in plain English, before running the code)*

In [ ]:
for key, value in me.items():
    print(f"{key}: {value}")

**📝 Reflect** — what did I actually see, and what did I learn?

*(write your reflection here, in your own words, after running)*

### 2e — Modify: build a new list using `.items()`

Now use what you just learned. The code below builds a *list of strings*, one per key-value pair. This is the kind of thing I'd do to summarize a dict for a print, log, or LLM prompt.

**🔮 Predict** — what do I think the next code cell will do?

*(write your prediction here, in plain English, before running the code)*

In [ ]:
summary_lines = []
for key, value in me.items():
    summary_lines.append(f"My {key} is {value}.")

for line in summary_lines:
    print(line)

**📝 Reflect** — what did I actually see, and what did I learn?

*(write your reflection here, in your own words, after running)*

💡 *Stuck or curious? Ask Claude inline:*

```python
print_llm_response("Explain the difference between .keys(), .values(), and .items() in plain English.")
```

**🧩 Synthesis** — When would I use `.keys()` vs `.values()` vs `.items()`? Write a one-sentence rule for each.

*(write your synthesis here in your own words)*

# Lesson 3: Nested Dictionaries

Real data is rarely flat. JSON responses, JSONL conversation files, configuration files — they're almost always **dictionaries containing other dictionaries**. The pattern shows up everywhere, and `extract_text` from this morning is exactly this: `entry` is a dict, `entry["message"]` is *also* a dict, and we have to dig two levels deep to get to what we want.

Here's a nested dictionary that mirrors the shape of a Claude Code JSONL entry (simplified):

In [ ]:
jsonl_entry = {
    "type": "assistant",
    "uuid": "abc-123",
    "message": {
        "role": "assistant",
        "model": "claude-sonnet-4-7",
        "content": "Hello Jen!"
    }
}

### 3a — Accessing a nested value

To reach `"Hello Jen!"`, I have to go through `"message"` first, then into `"content"`. Two lookups, chained.

**🔮 Predict** — what do I think the next code cell will do?

*(write your prediction here, in plain English, before running the code)*

In [ ]:
print(jsonl_entry["message"]["content"])

**📝 Reflect** — what did I actually see, and what did I learn?

*(write your reflection here, in your own words, after running)*

### 3b — What happens when an intermediate key is missing?

If the `"message"` key didn't exist, the lookup would crash *before* even getting to `"content"`. The error happens at the FIRST missing key in the chain.

**🔮 Predict** — what do I think the next code cell will do?

*(write your prediction here, in plain English, before running the code)*

In [ ]:
# This will crash — there is no key called "metadata"
print(jsonl_entry["metadata"]["created_at"])

**📝 Reflect** — what did I actually see, and what did I learn?

*(write your reflection here, in your own words, after running)*

### 3c — Chained `.get()` — the naive version (also crashes!)

You might think: *I'll just use `.get()` for safety!* Watch what happens:

**🔮 Predict** — what do I think the next code cell will do?

*(write your prediction here, in plain English, before running the code)*

In [ ]:
# Still crashes — but for a different reason this time
print(jsonl_entry.get("metadata").get("created_at"))

**📝 Reflect** — what did I actually see, and what did I learn?

*(write your reflection here, in your own words, after running)*

*Hint if you got surprised:* `.get("metadata")` returned `None` because the key was missing — and `None.get(...)` doesn't work, because `None` isn't a dictionary. **This is the trap we discussed in this morning's `extract_text` walkthrough.**

### 3d — The fix: default to an empty dict `{}`

If you give `.get()` a second argument of `{}` (an empty dict), then when the key is missing you get an empty dict back instead of `None`. And `{}.get("anything")` returns `None` safely — no crash.

**🔮 Predict** — what do I think the next code cell will do?

*(write your prediction here, in plain English, before running the code)*

In [ ]:
# Safe chained lookup — never crashes, even when keys are missing
print(jsonl_entry.get("metadata", {}).get("created_at"))

**📝 Reflect** — what did I actually see, and what did I learn?

*(write your reflection here, in your own words, after running)*

💡 *This is the exact pattern from `extract_text`:* `entry.get("message", {}).get("content")`. The `{}` is a safety net — if `"message"` is missing, you still get a dict to call `.get()` on.

### 3e — Modify: write a safe lookup yourself

Try writing a safe nested lookup for a key path that *does* exist. Predict what it returns first.

**🔮 Predict** — what do I think the next code cell will do?

*(write your prediction here, in plain English, before running the code)*

In [ ]:
# Safe lookup for the model name
model = jsonl_entry.get("message", {}).get("model")
print(model)

**📝 Reflect** — what did I actually see, and what did I learn?

*(write your reflection here, in your own words, after running)*

💡 *Stuck or curious? Ask Claude inline:*

```python
print_llm_response("Explain why entry.get('message', {}).get('content') works but entry.get('message').get('content') can crash.")
```

**🧩 Synthesis** — In my own words: why do I need to pass `{}` as a default to `.get()` when I'm about to chain another `.get()` after it? Connect this back to the `extract_text` function — what would happen if I removed the `{}` defaults from those chained lookups?

*(write your synthesis here in your own words)*

# Lesson 4: Dict vs List — Choosing the Right Container

Lists and dictionaries are both ways to hold a collection of values. But they answer different questions. Picking the wrong one makes your code awkward; picking the right one makes it obvious. This lesson is about *when* to reach for each.

### 4a — Lists are for ordered sequences

When the **order matters** and the items don't have unique labels, you want a list. *"What's the third item?"* makes sense for a list. It makes no sense for a dict.

**🔮 Predict** — what do I think the next code cell will do?

*(write your prediction here, in plain English, before running the code)*

In [ ]:
essay_drafts = ["The Values-Tool Problem", "Social Contract for AI", "Developmental Frame"]

print(f"Total drafts: {len(essay_drafts)}")
print(f"First draft: {essay_drafts[0]}")
print(f"Last draft: {essay_drafts[-1]}")

**📝 Reflect** — what did I actually see, and what did I learn?

*(write your reflection here, in your own words, after running)*

### 4b — Dicts are for labeled lookups

When you want to **look something up by name**, you want a dict. Order doesn't matter; the label does.

**🔮 Predict** — what do I think the next code cell will do?

*(write your prediction here, in plain English, before running the code)*

In [ ]:
essay_status = {
    "values-tool": "published",
    "social-contract": "drafting",
    "developmental-frame": "sketching"
}

print(f"Status of social-contract: {essay_status['social-contract']}")
print(f"Status of values-tool: {essay_status['values-tool']}")

**📝 Reflect** — what did I actually see, and what did I learn?

*(write your reflection here, in your own words, after running)*

### 4c — A list of dicts: when each item needs structure

Often you want *several* labeled items, in order — like a series of records. The natural shape is a **list of dicts**: each item is a labeled record, but the order is preserved.

**🔮 Predict** — what do I think the next code cell will do?

*(write your prediction here, in plain English, before running the code)*

In [ ]:
essays = [
    {"title": "The Values-Tool Problem", "status": "published", "word_count": 1200},
    {"title": "Social Contract for AI", "status": "drafting", "word_count": 400},
    {"title": "Developmental Frame", "status": "sketching", "word_count": 0}
]

for essay in essays:
    print(f"{essay['title']} — {essay['status']} ({essay['word_count']} words)")

**📝 Reflect** — what did I actually see, and what did I learn?

*(write your reflection here, in your own words, after running)*

💡 *This is exactly the shape of your `conversations.json` export from claude.ai — a **list** of **dicts**, where each dict is one conversation with labeled fields.*

**🧩 Synthesis** — Write a one-sentence rule for choosing list vs dict vs list-of-dicts. If I'm starting a new data structure tomorrow, what question do I ask myself first?

*(write your synthesis here in your own words)*

---

## 🎉 End of Module 04

You now know:
- `[]` vs `.get()` and when to reach for each
- `.get()` with a default value
- Iterating with `.keys()`, `.values()`, `.items()`
- Nested dicts and the `.get("key", {})` chaining pattern
- When to use a list, a dict, or a list of dicts

**Next:** Module 05 — Functions in depth. You've been *using* functions all along (`print`, `.get()`, `print_llm_response`). Next module: writing your own.